# E4 · Muestreo de identidades y reproducibilidad

**Estado:** protocolo prospectivo; no hay resultados hasta ejecutar y publicar.

**Objetivo y pregunta.** Medir si más identidades negativas por lote mejoran la adaptación visual y cuánto varía el resultado entre semillas.
**Hipótesis.** P=16, K=2 puede mejorar la separación entre individuos frente a P=8, K=4 con el mismo número de imágenes y pasos.
**Intervención.** Dos composiciones de lote, cada una con semillas 45, 46 y 47; seis candidatos fijados antes de entrenar. No se eligen variantes con test.
**Controles.** E3 seed 46, versión 20; CLIP, YOLO, cabeza E3, margen, tasa, épocas, paciencia y datos iguales. Se entrena LoRA q/v **y la proyección visual**, como hace realmente E4 histórico. Los pesos base de visión y la cabeza E3 permanecen congelados. La semilla ahora se fija **antes** de construir los adaptadores; el histórico no garantizaba esa condición.
**Muestra.** Corpus público refinado v2, train/val por identidad; test público inmutable v1 (v2 añade cajas) de 277 fotos/60 identidades, PetTrace v6 de 502 fotos, con galerías separadas. El loader verifica separación y commits.
**Selección.** Checkpoint por mAP macro de validación. Familia por media de sus tres mejores mAP de validación; empate favorece P=8. Candidato de esa familia por validación; empate favorece semilla menor. Se congelan ambas decisiones antes del test y se publican los seis candidatos, incluidos los que empeoren.
**Métricas.** Rank-1/2/5/10 y mAP, por especie/dominio y por semilla; media y desviación estándar descriptivas entre tres semillas. No confundir esta dispersión con incertidumbre poblacional. EER descriptiva sobre los mismos pares.
**Limitaciones.** Benchmark ya observado, sin separación de sesiones garantizada, recuperación cerrada de la misma especie y leave-one-out por foto; no demuestra búsqueda abierta ni rendimiento urbano. P=16 también reduce positivos por identidad: el contraste mide toda la composición del lote.

LoRA: [Hu et al.](https://arxiv.org/abs/2106.09685). Batch-hard y muestreo P×K: [Hermans et al.](https://arxiv.org/abs/1703.07737). CLIP: [Radford et al.](https://proceedings.mlr.press/v139/radford21a.html). YOLOv8: [Ultralytics](https://docs.ultralytics.com/models/yolov8/). La selección de lotes, semillas, presupuesto y reglas de selección es una decisión de esta investigación.

### Entorno
Paquete publicado fijado; se fija la semilla antes de `build_encoder` porque esta versión aún inicializa los adaptadores antes de la semilla interna de `fit`. No se cambia la implementación de la pérdida. La publicación es obligatoria y el registro se conserva activado solo para el candidato seleccionado, sin mover aliases.

In [1]:
%pip install -q --no-cache-dir "pettrace-ml[training,notebooks,tracking]==0.2.46"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import json, os, sys
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import mlflow
import numpy as np
import torch
from IPython.display import Markdown, display
from google.colab import userdata
import pettrace
from pettrace.tracking import PRODUCTION_URI, source_identity

if not (3, 11) <= sys.version_info[:2] < (3, 14) or sys.platform != "linux":
    raise RuntimeError("E4 requires Colab Linux with Python 3.11–3.13")
if pettrace.__version__ != "0.2.46":
    raise RuntimeError("Install PetTrace 0.2.46 before continuing")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; a T4 is sufficient")
print("Access [1/2] Loading Colab Secrets…")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise TypeError(f"Colab Secret {name} must be a non-empty text value")
    os.environ[name] = value
print("Access [2/2] Credentials available.")
EXPERIMENT, DATASET = "E4", "v6"
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
TRAINING_DATASET_VERSION, SOURCE_SAMPLE_PCT = "v2", 100
SEEDS, BATCHES = (45, 46, 47), ((8, 4), (16, 2))
CLIP_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
BASE_MODEL = "models:/pettrace-embedder/2"
LOG_TO_MLFLOW = True
REGISTER_E4 = True
RUN_LABEL = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
REPORT = Path("/content/reports") / "e4_sampling_study" / RUN_LABEL
REPORT.mkdir(parents=True)
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
SOURCE_IDENTITY = source_identity()
SOURCE_SHA256 = SOURCE_IDENTITY["source_sha256"]
mlflow.set_tracking_uri(PRODUCTION_URI); mlflow.set_registry_uri(PRODUCTION_URI)
print("Package:", pettrace.__version__, "| GPU:", torch.cuda.get_device_name(0), "| Report:", REPORT)
STUDY = {"id": RUN_LABEL, "seeds": list(SEEDS), "batches": list(BATCHES),
         "trainables": "LoRA q/v and visual projection; frozen E3 head",
         "initialization_seeded": True, "selection": "mean validation mAP per family, then best validation seed",
         "tie_break": "lower P, then lower seed", "test_used_for_selection": False}
(REPORT / "study-protocol.json").write_text(json.dumps(STUDY, indent=2))

Access [1/2] Loading Colab Secrets…
Access [2/2] Credentials available.
Package: 0.2.46 | GPU: NVIDIA L4 | Report: /content/reports/e4_sampling_study/20261118T053018Z-9d7ddbb5


414

## Fase 1. Preparación y preprocesamiento de datos
Se cargan snapshots ya preparados; no se limpia ni redivide el corpus. Las cajas YOLO se verifican contra el detector E2. Las fotografías privadas solo se utilizan para evaluación separada.

In [3]:
from pettrace.data import training
from pettrace.data.volume import training_volume

print("Dataset [1/2] Downloading pinned public v2 shards and verifying hashes…")
prepared = training(version=TRAINING_DATASET_VERSION,
    cache_dir="/content/datasets/pettrace-refined", token=os.environ["HF_TOKEN"])
manifest_rows = prepared.rows
training_source = prepared.provenance
coverage_metrics, coverage_params = training_volume(prepared, SOURCE_SAMPLE_PCT, REPORT)
(REPORT / "transport-equivalence.json").write_text(json.dumps(training_source, indent=2))
print("Dataset [2/2] Verified:", prepared.fingerprint, "| photos:", len(manifest_rows),
      "| identities:", prepared.summary["retained_identities"],
      "| commits:", [source.commit[:12] for source in prepared.sources])

Dataset [1/2] Downloading pinned public v2 shards and verifying hashes…
Dataset: verifying and materializing alessandro54/labeled-cats-in-the-wild-refined@581e44a04fd4…
Dataset: lcw ready (131,975 photos).
Dataset: verifying and materializing alessandro54/dogs-of-the-world-refined@9fdd6e5f7eaa…
Dataset: dogs-world ready (61,851 photos).
Dataset [2/2] Verified: d0ca5c0b1bed8844f943b9fd355ce912524797d0dc71b50c7a1624e39f49fe58 | photos: 193826 | identities: 42113 | commits: ['581e44a04fd4', '9fdd6e5f7eaa']


In [4]:
from pettrace.tracking import production_context
from pettrace.experiments.e3 import load_benchmarks
production_context()
benchmarks = load_benchmarks(prepared, REPORT, private_version=DATASET, private_commit=DATASET_COMMIT)
private_ds, public_ds, PUBLIC_REPORT = benchmarks.private, benchmarks.public, benchmarks.public_report
print("Public test:", len(public_ds.photos), "photos | PetTrace:", len(private_ds.photos), "photos")

Loading immutable benchmarks:   0%|          | 0/3 [00:00<?, ?step/s]

Benchmarks ready in 69.8s: 277 public · 502 PetTrace.
Public test: 277 photos | PetTrace: 502 photos


## Fase 2. Representación y entrenamiento
Se mantiene la intervención visual histórica (LoRA más proyección) y se verifica el conjunto exacto de pesos entrenables. Cada corrida conserva configuración, lista de parámetros, paridad, checkpoint, curvas y tiempos. Las seis se entrenan antes de observar métricas de test. Un resumen existente permite retomar la exportación sin reentrenar; un fallo deja los archivos locales disponibles.

In [5]:
from pettrace.evaluation.core import load_model
from pettrace.experiments.e4 import load_control_head
from pettrace.training.lora import Config, PreparedPreprocessor, build_encoder, control_equivalence

base_model = load_model(BASE_MODEL)
if not base_model.golden_ok or base_model.pipeline.detector is None:
    raise RuntimeError("The fixed E2 base failed verification")

head, control_receipt = load_control_head(REPORT)
preprocessor = PreparedPreprocessor(base_model.pipeline, prepared)

training_rows = [row for row in manifest_rows if row["split"] in {"train", "val"}]
train_rows = [row for row in training_rows if row["split"] == "train"]
gate = [
    next(row for row in sorted(train_rows, key=lambda item: item["sha256"])
         if row["species"] == species)
    for species in ("cat", "dog")
]

checked = preprocessor.validate(gate)
pixels = preprocessor.pixels(gate)
e2 = base_model.pipeline.session.run(
    [base_model.pipeline.output_name],
    {base_model.pipeline.input_name: pixels},
)[0]

In [6]:
import copy, gc, shutil, time, yaml
from pettrace.training.triplet import seed_all
from pettrace.training.lora import fit, trainable_parameters, export_encoder, load_adapter
from pettrace.experiments.e3 import select_validation_candidate

candidates = []
for p, k in BATCHES:
    for seed in SEEDS:
        run_report = REPORT / f"p{p}-k{k}-seed{seed}"
        run_report.mkdir(exist_ok=True)
        for filename in ("dataset-coverage.json", "sources.json", "transport-equivalence.json", "study-protocol.json"):
            shutil.copy2(REPORT / filename, run_report / filename)
        config = Config(seed=seed, identities_per_batch=p, images_per_identity=k)
        seed_all(seed)
        encoder, adapted_layers = build_encoder(copy.deepcopy(head), CLIP_REVISION, config=config)
        parity = control_equivalence(encoder, pixels, e2)
        trainables = {name: list(value.shape) for name, value in encoder.named_parameters() if value.requires_grad}
        assert "visual_projection.weight" in trainables
        assert all(name == "visual_projection.weight" or name.endswith((".a", ".b")) for name in trainables)
        run_config = {"experiment": EXPERIMENT, "training": asdict(config), "study": STUDY,
                      "control": control_receipt, "clip_revision": CLIP_REVISION,
                      "source": SOURCE_IDENTITY, "trainable_parameters": trainable_parameters(encoder),
                      "trainable_names_and_shapes": trainables, "log_to_mlflow": True,
                      "register_selected_model": REGISTER_E4}
        (run_report / "training-config.json").write_text(json.dumps(run_config, indent=2))
        (run_report / "control-equivalence.json").write_text(json.dumps(parity, indent=2))
        summary_path = run_report / "training-summary.json"
        if summary_path.exists():
            training_summary = json.loads(summary_path.read_text())
            assert training_summary["config"] == asdict(config), "Stored configuration differs"
            load_adapter(encoder, run_report / "adapter.pt")
        else:
            started = time.monotonic()
            print(f"Training P={p}, K={k}, seed={seed}; validation only", flush=True)
            encoder, training_summary = fit(encoder, training_rows, preprocessor, run_report,
                                            config=config, device="cuda")
            training_summary["elapsed_seconds"] = time.monotonic() - started
            summary_path.write_text(json.dumps(training_summary, indent=2))
        model_dir = run_report / "model"
        if not (model_dir / "golden.json").exists():
            export_encoder(base_model.dir, encoder, model_dir,
                model_version=f"clip-vit-b32-crop-lora-projection-p{p}k{k}-seed{seed}@study1",
                verification_pixels=pixels)
            spec = yaml.safe_load((model_dir / "pipeline.yaml").read_text())
            spec["adaptation"].update(method="LoRA and visual projection",
                visual_projection_trainable=True, base="frozen CLIP vision base and frozen E3-100 head")
            (model_dir / "pipeline.yaml").write_text(yaml.safe_dump(spec, sort_keys=False))
        candidates.append({"p": p, "k": k, "seed": seed, "report": run_report,
            "model_dir": model_dir, "config": config, "training_summary": training_summary,
            "best_val_mAP": training_summary["best_val_mAP"], "best_epoch": training_summary["best_epoch"]})
        del encoder
        gc.collect(); torch.cuda.empty_cache()
        print(f"Finished training P={p}, seed={seed}: validation mAP={training_summary['best_val_mAP']:.6f}", flush=True)

assert len(candidates) == 6
family_scores = {p: float(np.mean([c["best_val_mAP"] for c in candidates if c["p"] == p])) for p, _ in BATCHES}
selected_p = max(family_scores, key=lambda p: (family_scores[p], -p))
selected = select_validation_candidate([c for c in candidates if c["p"] == selected_p])
selection = {"selected_before_test_evaluation": True, "family_validation_means": family_scores,
             "selected_p": selected_p, "selected_seed": selected["seed"], "study": STUDY,
             "candidates": [{key: c[key] for key in ("p", "k", "seed", "best_epoch", "best_val_mAP")} for c in candidates]}
(REPORT / "selection.json").write_text(json.dumps(selection, indent=2))
for candidate in candidates:
    shutil.copy2(REPORT / "selection.json", candidate["report"] / "selection.json")
print("Selection frozen before test:", selection)

Training P=8, K=4, seed=45; validation only


Training seed 45:   0%|          | 0/30 [00:00<?, ?epoch/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/clip/modeling_clip.py:243: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not interpolate_pos_encoding and (height != self.image_size or width != self.image_size):


Finished training P=8, seed=45: validation mAP=0.817579
Training P=8, K=4, seed=46; validation only


Training seed 46:   0%|          | 0/30 [00:00<?, ?epoch/s]

Finished training P=8, seed=46: validation mAP=0.847558
Training P=8, K=4, seed=47; validation only


... (4 outputs left out)

Training seed 46:   0%|          | 0/30 [00:00<?, ?epoch/s]

Finished training P=16, seed=46: validation mAP=0.844367
Training P=16, K=2, seed=47; validation only


Training seed 47:   0%|          | 0/30 [00:00<?, ?epoch/s]

Finished training P=16, seed=47: validation mAP=0.831400
Selection frozen before test: {'selected_before_test_evaluation': True, 'family_validation_means': {8: 0.8339350415038922, 16: 0.8387905205483156}, 'selected_p': 16, 'selected_seed': 46, 'study': {'id': '20261118T053018Z-9d7ddbb5', 'seeds': [45, 46, 47], 'batches': [(8, 4), (16, 2)], 'trainables': 'LoRA q/v and visual projection; frozen E3 head', 'initialization_seeded': True, 'selection': 'mean validation mAP per family, then best validation seed', 'tie_break': 'lower P, then lower seed', 'test_used_for_selection': False}, 'candidates': [{'p': 8, 'k': 4, 'seed': 45, 'best_epoch': 9, 'best_val_mAP': 0.8175792674412365}, {'p': 8, 'k': 4, 'seed': 46, 'best_epoch': 23, 'best_val_mAP': 0.8475579872967853}, {'p': 8, 'k': 4, 'seed': 47, 'best_epoch': 25, 'best_val_mAP': 0.8366678697736549}, {'p': 16, 'k': 2, 'seed': 45, 'best_epoch': 26, 'best_val_mAP': 0.8406044437944513}, {'p': 16, 'k': 2, 'seed': 46, 'best_epoch': 21, 'best_val_mAP'

## Fase 3. Evaluación y análisis
Se evalúan los seis candidatos sin modificar la selección ya guardada. Rank-k agrega el máximo por identidad; AP/mAP ordena fotografías. Se publican todos los resultados en MLflow antes de registrar el seleccionado. Si falla esta celda, se reejecuta sin repetir la fase de entrenamiento; los recibos existentes evitan duplicar evaluaciones terminadas en esta sesión. Los resultados y limitaciones se completan con la evidencia real, sin asegurar mejoras de antemano.

In [7]:
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import save
from pettrace.experiments.e3 import publish_results, register_candidate
from pettrace.tracking import save_provenance

for candidate in candidates:
    run_report = candidate["report"]
    run_public_report = run_report / "public_test"
    run_public_report.mkdir(exist_ok=True)
    shutil.copy2(PUBLIC_REPORT / "dataset-selection.json", run_public_report / "dataset-selection.json")
    e4_model = load_model(str(candidate["model_dir"]))
    if not e4_model.golden_ok:
        raise RuntimeError("Exported E4 failed golden verification")
    receipt = run_report / "publication.json"
    if receipt.exists() and json.loads(receipt.read_text()).get("published"):
        publication = json.loads(receipt.read_text())
        client = mlflow.MlflowClient()
        assert all(client.get_run(publication[key]).info.status == "FINISHED" for key in ("run_id", "public_run_id"))
    else:
        public_ev = evaluate(public_ds, e4_model.pipeline, zero_shot=True)
        private_ev = evaluate(private_ds, e4_model.pipeline, zero_shot=True)
        save(public_ev, e4_model, str(run_public_report))
        save(private_ev, e4_model, str(run_report))
        save_provenance(run_public_report, e4_model)
        publication = publish_results(public_ev, private_ev, e4_model, run_report, run_public_report, SOURCE_SHA256,
            experiment=EXPERIMENT, source_sample_pct=SOURCE_SAMPLE_PCT, config=candidate["config"],
            training_summary=candidate["training_summary"], prepared_fingerprint=prepared.fingerprint,
            training_source=training_source, coverage_metrics=coverage_metrics,
            coverage_params=coverage_params, private_dataset_version=DATASET)
    candidate["publication"] = publication
    for run_id in (publication["run_id"], publication["public_run_id"]):
        mlflow.MlflowClient().set_tag(run_id, "study_id", RUN_LABEL)
        mlflow.MlflowClient().set_tag(run_id, "adaptation", "lora_and_visual_projection")
    print("Published:", candidate["p"], candidate["seed"], publication, flush=True)
    del e4_model
    gc.collect()

if REGISTER_E4:
    publication = selected["publication"]
    registration = register_candidate(selected["report"], publication["run_id"], publication["public_run_id"], experiment=EXPERIMENT)
    print("Selected candidate version:", registration["version"], "| no alias promoted")

🏃 View run E4 · clip-vit-b32-crop-lora-projection-p8k4-seed45@study1 on lcw+dow-refined@v2 at: https://ml.pettrace.app/#/experiments/3/runs/115e2ae3410e477c900af9c230ca9987
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3
🏃 View run E4 · clip-vit-b32-crop-lora-projection-p8k4-seed45@study1 on pettrace-reid@v6 at: https://ml.pettrace.app/#/experiments/3/runs/aede0c3f2dae4678910e0e4abece0ec9
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3
Published: 8 45 {'run_id': 'aede0c3f2dae4678910e0e4abece0ec9', 'public_run_id': '115e2ae3410e477c900af9c230ca9987'}
🏃 View run E4 · clip-vit-b32-crop-lora-projection-p8k4-seed46@study1 on lcw+dow-refined@v2 at: https://ml.pettrace.app/#/experiments/3/runs/efb248ea8c78426cac99e4228314b0cf
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3
🏃 View run E4 · clip-vit-b32-crop-lora-projection-p8k4-seed46@study1 on pettrace-reid@v6 at: https://ml.pettrace.app/#/experiments/3/runs/9cf78b21b3e04334bf738a29bcbcc167
🧪 View exp

2026/11/18 08:21:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: pettrace-embedder, version 24


Selected candidate version: 24 | no alias promoted


In [8]:
results = []
for candidate in candidates:
    for domain, folder in (("public", candidate["report"] / "public_test"), ("private", candidate["report"])):
        summary = json.loads((folder / "summary.json").read_text())
        results.append({"domain": domain, "p": candidate["p"], "k": candidate["k"], "seed": candidate["seed"],
                        "best_val_mAP": candidate["best_val_mAP"], "metrics": summary["global"],
                        "publication": candidate["publication"]})
(REPORT / "study-results.json").write_text(json.dumps(results, indent=2))
for domain in ("public", "private"):
    for p, k in BATCHES:
        group = [r for r in results if r["domain"] == domain and r["p"] == p]
        print(domain, f"P={p}, K={k}", {metric: {"mean": float(np.mean([r['metrics'][metric] for r in group])),
              "sd_seeds": float(np.std([r['metrics'][metric] for r in group], ddof=1))} for metric in ("rank1", "rank2", "mAP")})
for run_id in (selected["publication"]["run_id"], selected["publication"]["public_run_id"]):
    mlflow.MlflowClient().log_artifact(run_id, str(REPORT / "study-results.json"), artifact_path="study")
    mlflow.MlflowClient().log_artifact(run_id, str(REPORT / "selection.json"), artifact_path="study")
print("Study complete: six candidates, both domains published. Selected only by validation.")

public P=8, K=4 {'rank1': {'mean': 0.8555956678700362, 'sd_seeds': 0.018758673006161138}, 'rank2': {'mean': 0.9602888086642599, 'sd_seeds': 0.0072202166064981865}, 'mAP': {'mean': 0.8207898608812364, 'sd_seeds': 0.020953682142998625}}
public P=16, K=2 {'rank1': {'mean': 0.8640192539109507, 'sd_seeds': 0.011604874561965087}, 'rank2': {'mean': 0.9602888086642599, 'sd_seeds': 0.009551448776406495}, 'mAP': {'mean': 0.8186483206414992, 'sd_seeds': 0.01696798917932919}}
private P=8, K=4 {'rank1': {'mean': 0.8426294820717132, 'sd_seeds': 0.017366131249166014}, 'rank2': {'mean': 0.9349269588313414, 'sd_seeds': 0.006403486561084358}, 'mAP': {'mean': 0.7178163485200257, 'sd_seeds': 0.0148520860989919}}
private P=16, K=2 {'rank1': {'mean': 0.8419654714475432, 'sd_seeds': 0.004146745018856834}, 'rank2': {'mean': 0.9462151394422311, 'sd_seeds': 0.009128636842541567}, 'mAP': {'mean': 0.7210094475486261, 'sd_seeds': 0.0124550960498678}}
Study complete: six candidates, both domains published. Selected

### Resultados registrados — 18/11/2026

**Ejecutado y publicado:** seis candidatos, doce evaluaciones FINISHED. Selección por validación: P16×K2, semilla 46; versión 24 candidata, sin aliases promovidos.

El seleccionado obtiene 238/277 (85,92 %, mAP 0,8336) en público y 425/502 (84,66 %, mAP 0,7092) en PetTrace. Frente al E4 histórico son +1 y −8 aciertos: no es una mejora general. DINOv2-L congelado obtiene 255/277 y 476/502.

La exposición efectiva varió por parada temprana (14–30 épocas); solo el presupuesto máximo fue común. Tres semillas no cuantifican generalización poblacional y no se garantizó separación entre sesiones. El informe conserva todos los resultados, incluidos los negativos.

[Informe completo](https://ml.pettrace.app/#/experiments/3/runs/c230a966311144f68a26522858a8c3d4) en los artefactos `supplement` de la corrida seleccionada.